# 🧹 Smart Business Intelligence - Data Cleaning & Preprocessing

**Projet :** Smart Business Intelligence & Sales Analytics Platform  
**Auteur :** Chaima Debchi  
**Objectif :** Nettoyer, structurer et enrichir les données brutes de transactions commerciales pour alimenter les analyses statistiques, les modèles de Machine Learning et le dashboard interactif.

---

## 📋 Sommaire
1. Chargement des données brutes et première inspection
2. Détection des problèmes de qualité des données
3. Séparation des tables imbriquées et extraction des Retours (`Returns`)
4. Traitement des valeurs manquantes et imputation justifiée
5. Standardisation des types de données (Dates, Numériques, Catégories)
6. Ingénierie des variables (*Feature Engineering* métier)
7. Validation finale et sauvegarde des données nettoyées

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration de l'affichage
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

raw_path = Path('../data/raw/sample_superstore.csv')
processed_path = Path('../data/processed/cleaned_superstore.csv')
print('Fichier brut existe :', raw_path.exists())

## 1. Chargement des données brutes
Le dataset utilise l'encodage `ISO-8859-1` (Latin-1). Chargeons le fichier brut et inspectons ses dimensions.

In [ ]:
df_raw = pd.read_csv(raw_path, encoding='ISO-8859-1')
print(f'Dimensions du fichier brut : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes')
df_raw.head(3)

## 2. Détection des problèmes de qualité

Analysons les valeurs manquantes et la structure du fichier.

In [ ]:
print('=== VALEURS MANQUANTES PAR COLONNE ===')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

> **Observation :** Les colonnes ont 806 valeurs nulles à partir de la ligne 9994. En inspectant ces lignes, on constate que le fichier contient des tables annexes (`People` et `Returns`) concaténées à la suite de la table principale `Orders`.
>
> **Décision métier :**
> 1. Conserver les lignes 0 à 9993 comme table de commandes principale.
> 2. Extraire la liste des `Order ID` retournés pour créer une variable binaire `Returned` (`Yes`/`No`).

In [ ]:
# 1. Extraction de la table des commandes principale
df_orders = df_raw.iloc[:9994].copy()

# 2. Extraction des retours
df_returns = df_raw.iloc[10000:].dropna(subset=['Order ID']).copy()
returned_orders = set(df_returns['Order ID'].unique())
df_orders['Returned'] = df_orders['Order ID'].isin(returned_orders).map({True: 'Yes', False: 'No'})

print(f'Nombre de commandes retournées identifiées : {len(returned_orders)}')
print(df_orders['Returned'].value_counts())

## 3. Traitement des valeurs manquantes dans `Postal Code`
Regardons les lignes où `Postal Code` est nul dans `df_orders`.

In [ ]:
missing_zip = df_orders[df_orders['Postal Code'].isnull()]
print(f'Nombre de lignes sans code postal : {len(missing_zip)}')
print(missing_zip[['City', 'State', 'Region']].drop_duplicates())

> **Décision métier :** Toutes les valeurs manquantes appartiennent à **Burlington, Vermont**.
> Le code postal officiel de Burlington (VT) est `05401`. Le zéro initial a été effacé par un export tableur.
> Nous imputons donc `05401` au lieu de supprimer ces 11 lignes de chiffre d'affaires légitime.

In [ ]:
df_orders.loc[df_orders['Postal Code'].isnull() & (df_orders['City'] == 'Burlington'), 'Postal Code'] = 5401
df_orders['Postal Code'] = df_orders['Postal Code'].astype(int).astype(str).str.zfill(5)
print('Valeurs manquantes restantes dans Postal Code :', df_orders['Postal Code'].isnull().sum())

## 4. Conversion des types et Dates
Convertissons `Order Date` et `Ship Date` au format datetime pour permettre des analyses temporelles.

In [ ]:
df_orders['Order Date'] = pd.to_datetime(df_orders['Order Date'], format='%m/%d/%Y')
df_orders['Ship Date'] = pd.to_datetime(df_orders['Ship Date'], format='%m/%d/%Y')
print('Plage de dates : de', df_orders['Order Date'].min(), 'à', df_orders['Order Date'].max())

## 5. Ingénierie des variables (Feature Engineering)

Créons des variables analytiques indispensables pour l'EDA et les modèles ML :
- **Temporelles :** Année, Mois, Nom du mois, Année-Mois, Trimestre, Jour de semaine.
- **Logistique :** Délai de livraison (`Ship Date - Order Date`).
- **Financières :** Marge bénéficiaire (`Profit / Sales * 100`), Prix unitaire (`Sales / Quantity`), Montant de remise.

In [ ]:
# Variables temporelles
df_orders['Order Year'] = df_orders['Order Date'].dt.year
df_orders['Order Month'] = df_orders['Order Date'].dt.month
df_orders['Order Month Name'] = df_orders['Order Date'].dt.month_name()
df_orders['Order YearMonth'] = df_orders['Order Date'].dt.strftime('%Y-%m')
df_orders['Order Quarter'] = df_orders['Order Date'].dt.to_period('Q').astype(str)
df_orders['Order DayOfWeek'] = df_orders['Order Date'].dt.day_name()

# Délai logistique
df_orders['Delivery Days'] = (df_orders['Ship Date'] - df_orders['Order Date']).dt.days

# Variables financières
df_orders['Profit Margin'] = ((df_orders['Profit'] / df_orders['Sales']) * 100).round(2)
df_orders['Unit Price'] = (df_orders['Sales'] / df_orders['Quantity']).round(2)
df_orders['Discount Amount'] = ((df_orders['Sales'] / (1 - df_orders['Discount'])) - df_orders['Sales']).round(2)

if 'Row ID' in df_orders.columns:
    df_orders['Row ID'] = df_orders['Row ID'].astype(int)

print('Nouvelles dimensions :', df_orders.shape)

## 6. Contrôle qualité final (Sanity Checks)
Vérifions qu'il ne reste aucune incohérence (valeurs manquantes, délais négatifs, doublons).

In [ ]:
print('1. Nombre total de valeurs nulles :', df_orders.isnull().sum().sum())
print('2. Doublons complets :', df_orders.duplicated().sum())
print('3. Délais de livraison négatifs :', (df_orders['Delivery Days'] < 0).sum())
print('4. Statistiques des délais (jours) : min =', df_orders['Delivery Days'].min(), ', max =', df_orders['Delivery Days'].max())
print('5. Chiffre d\'affaires total :', round(df_orders['Sales'].sum(), 2), '$')
print('6. Bénéfice total :', round(df_orders['Profit'].sum(), 2), '$')

## 7. Sauvegarde du jeu de données nettoyé

In [ ]:
processed_path.parent.mkdir(parents=True, exist_ok=True)
df_orders.to_csv(processed_path, index=False, encoding='utf-8')
print(f'Jeu de données nettoyé sauvegardé avec succès dans : {processed_path}')
print(f'Taille du fichier : {processed_path.stat().st_size / 1024:.1f} KB')